A basic neural network library should have the following features:

<b>Create NN:</b>

Inits numpy arrays for weights and biases and stores them all in an appropriate data structure

<b>Activation Function:</b>

Applies a non-linear function (e.g. sigmoid, ReLU, softmax) to a layer's output during forward propagation, and provides its derivative for use in backpropagation

<b>Loss/Cost Function:</b>

Computes the error between the network's output and the expected output (e.g. MSE, cross-entropy), and provides its derivative to seed backpropagation

<b>Forward propogation:</b>

Forward propogates the input through the layers, applying weights, biases, and activation functions, to get an output

<b>Back propogation:</b>

Backpropogates the gradient of the loss with respect to the output back through the layers to get the gradient with respect to each weight and bias

<b>Training Cycle:</b>

Forward propogates input through an NN in batches and updates the weights and biases accordingly

<b>Testing Cycle:</b>

Runs the forward prop only on test data to see the accuracy

<b>Save weights:</b>

Saves the weights as a .csv

<b>Load weights:</b>

Loads the weights from a .csv

In [3]:
import numpy as np

In [26]:
class NeuralNetwork:


    def __init__(self, layer_sizes:list, activation_functions:list, data_type = np.float32):

        activation_functions_dict = {
            'ReLU': self.ReLU,
            'LeakyReLU': self.LeakyReLU,
            'Logistic': self.Logistic
        }

        derivative_functions_dict = {
            'ReLU': self.ReLU_derivative,
            'LeakyReLU': self.LeakyReLU_derivative,
            'Logistic': self.Logistic_derivative
        }

        self.activation_functions_names = activation_functions
        self.layer_sizes = layer_sizes
        self.layer_weights = [np.ones((layer_sizes[i+1], layer_sizes[i]), dtype=data_type) for i in range(len(layer_sizes) - 1) ]
        self.layer_biases = [np.zeros(i, dtype=data_type) for i in layer_sizes[1:]]
        self.length = len(layer_sizes)

        self.activation_functions = [activation_functions_dict.get(name) for name in activation_functions]  
        self.activation_functions_derivatives = [derivative_functions_dict.get(name) for name in activation_functions]


    def __repr__(self):

        text = "Neural Network\n"
        text += f"Input layer size: {self.layer_sizes[0]}\n"
        for i in range(self.length - 1):
            text += f"Layer {i+1}: Activation function: {self.activation_functions[i]}, Layer size: {self.layer_sizes[i]} -> {self.layer_sizes[i+1]}\n"
        text += f"Output layer size: {self.layer_sizes[-1]}\n"
        return text

    def forward_propogate(self, input_layer:np.array) -> np.array:

        forward_propogation_layers = [input_layer]
        for i in range(self.length - 1):
            temporary_layer = self.layer_weights[i] @ forward_propogation_layers[-1] + self.layer_biases[i]
            forward_propogation_layers.append(temporary_layer)

        return forward_propogation_layers

    def backward_propogate(self, forward_propogation_layers:np.array, target_output:np.array, learning_rate:float, batch_size:int):
    
        # Calculate the error at the output layer
        output_error = forward_propogation_layers[-1] - target_output

        # Backpropagate the error through the network
        for i in reversed(range(self.length - 1)):
            # Calculate the gradient of the activation function
            if self.activation_functions[i] == 'ReLU':
                activation_gradient = self.ReLU_derivative(forward_propogation_layers[i+1])
            elif self.activation_functions[i] == 'LeakyReLU':
                activation_gradient = self.LeakyReLU_derivative(forward_propogation_layers[i+1], k=0.01)
            elif self.activation_functions[i] == 'Logistic':
                activation_gradient = self.Logistic_derivative(forward_propogation_layers[i+1], k=1.0)
            else:
                raise ValueError(f"Unsupported activation function: {self.activation_functions[i]}")

            # Calculate the delta for the current layer
            delta = output_error * activation_gradient

            # Update weights and biases
            self.layer_weights[i] -= (learning_rate / batch_size) * np.outer(delta, forward_propogation_layers[i])
            self.layer_biases[i] -= (learning_rate / batch_size) * delta

            # Calculate the error for the next layer (if not at input layer)
            if i > 0:
                output_error = self.layer_weights[i].T @ delta
        


    def ReLU(self, layer:np.array):
        transformed_layer = np.copy(layer)
        transformed_layer[layer < 0] = 0
        return transformed_layer

    def ReLU_derivative(self, layer:np.array):
        derivative_layer = np.copy(layer)
        derivative_layer[layer < 0] = 0
        derivative_layer[layer >= 0] = 1
        return derivative_layer

    def LeakyReLU(self, layer:np.array, k:np.float16):
        transformed_layer = np.copy(layer)
        transformed_layer[layer < 0] = k * transformed_layer[layer < 0]
        return transformed_layer
    
    def LeakyReLU_derivative(self, layer:np.array, k:np.float16):
        derivative_layer = np.copy(layer)
        derivative_layer[layer < 0] = k
        derivative_layer[layer >= 0] = 1
        return derivative_layer

    def Logistic(self, layer:np.array, k:float):
        transformed_layer = np.copy(layer)
        return (1 / (1 + np.exp(-k * layer)))

    def Logistic_derivative(self, layer:np.array, k:float):
        transformed_layer = np.copy(layer)
        logistic_layer = self.Logistic(transformed_layer, k)
        return k * logistic_layer * (1 - logistic_layer)

    def save_NN_data(self, file_weights="weights.npz", file_biases="biases.npz"):
        np.savez(file_weights, *self.layer_weights)
        np.savez(file_biases, *self.layer_biases)

    def load_NN_data(self, file_weights="weights.npz", file_biases="biases.npz"):
        weights_data = np.load(file_weights)
        biases_data = np.load(file_biases)

        self.layer_weights = [weights_data[f'arr_{i}'] for i in range(len(weights_data.files))]
        self.layer_biases = [biases_data[f'arr_{i}'] for i in range(len(biases_data.files))]
        self.layer_sizes = [self.layer_weights[0].shape[1]] + [self.layer_biases[i].shape[0] for i in range(len(self.layer_biases))]
        self.length = len(self.layer_sizes)


In [30]:
NN = NeuralNetwork(layer_sizes=[3, 6, 2], activation_functions=["ReLU", "ReLU", "ReLU"])
NN.save_NN_data("saved/weight.npz", "saved/bias.npz")


anotherNN = NeuralNetwork(layer_sizes=[3], activation_functions=["ReLU"], data_type=np.float32)
anotherNN.load_NN_data("saved/weight.npz", "saved/bias.npz")